# 14. Performance Optimization

As datasets grow larger, writing efficient Pandas code becomes important. Using the wrong approach (like Python loops) on a large DataFrame can make code hundreds of times slower than necessary.

## Setup

We'll use our familiar **customers** DataFrame, and also generate a larger synthetic dataset to properly demonstrate performance differences.

In [1]:
import pandas as pd
import numpy as np
import time
customers = pd.DataFrame({
    "Name": ["Hema", "Chitra", "Koushi", "Subha", "Swathi"],
    "Age": [25, 27, 30, 28, 24],
    "City": ["Chennai", "Delhi", "Bangalore", "Mumbai", "Chennai"],
    "Salary": [50000, 55000, 61000, 58000, 62000]
})
customers

,Name,Age,City,Salary
0,Hema,25,Chennai,50000
1,Chitra,27,Delhi,55000
2,Koushi,30,Bangalore,61000
3,Subha,28,Mumbai,58000
4,Swathi,24,Chennai,62000


In [2]:
np.random.seed(42)
n = 1_000_000
big_df = pd.DataFrame({
    "Age": np.random.randint(18, 60, size=n),
    "Salary": np.random.randint(20000, 150000, size=n),
    "City": np.random.choice(["Chennai", "Delhi", "Bangalore", "Mumbai"], size=n)
})
big_df.shape

(1000000, 3)

## 1. Vectorization

Vectorization means applying an operation to an entire Series/array at once (using optimized C code under the hood) instead of looping through elements one by one in Python. Pandas and NumPy operations are vectorized by default.

**Syntax:**

```
DataFrame['col'] + DataFrame['col2']
DataFrame['col'] * 2
```

In [3]:
start = time.time()
big_df["Bonus"] = big_df["Salary"] * 0.10
vectorized_time = time.time() - start
print(f"Vectorized time: {vectorized_time:.4f} seconds")
big_df.head()

Vectorized time: 0.0341 seconds


,Age,Salary,City,Bonus
0,56,126467,Delhi,12646.7
1,46,49112,Delhi,4911.2
2,32,81452,Mumbai,8145.2
3,25,140582,Delhi,14058.2
4,38,97516,Mumbai,9751.6


**Example:**

A company calculating a 10% bonus for every employee's salary can do this in a single vectorized line, instead of looping through each employee record individually.

In [4]:
customers["Bonus"] = customers["Salary"] * 0.10
customers

,Name,Age,City,Salary,Bonus
0,Hema,25,Chennai,50000,5000.0
1,Chitra,27,Delhi,55000,5500.0
2,Koushi,30,Bangalore,61000,6100.0
3,Subha,28,Mumbai,58000,5800.0
4,Swathi,24,Chennai,62000,6200.0


### AI/ML Usage

Vectorized feature engineering (scaling, ratios, transformations) is essential when preparing large training datasets, since ML pipelines often process millions of rows.

## 2. apply() vs Loops

**apply()** runs a function along an axis of a DataFrame or Series. It is more readable and often faster than a plain Python **for** loop, but it is still generally slower than a fully vectorized operation.

**Syntax:**

```
Series.apply(function)
DataFrame.apply(function, axis=1)
```

In [5]:
start = time.time()
bonus_loop = []
for salary in big_df["Salary"][:100_000]:
    bonus_loop.append(salary * 0.10)
loop_time = time.time() - start
print(f"Loop time (100k rows): {loop_time:.4f} seconds")

Loop time (100k rows): 0.0896 seconds


In [6]:
start = time.time()
bonus_apply = big_df["Salary"][:100_000].apply(lambda x: x * 0.10)
apply_time = time.time() - start
print(f"apply() time (100k rows): {apply_time:.4f} seconds")

apply() time (100k rows): 0.0991 seconds


In [7]:
start = time.time()
bonus_vectorized = big_df["Salary"][:100_000] * 0.10
vector_time = time.time() - start
print(f"Vectorized time (100k rows): {vector_time:.4f} seconds")
print("\nSpeed comparison:")
print(f"Loop:        {loop_time:.4f}s")
print(f"apply():     {apply_time:.4f}s")
print(f"Vectorized:  {vector_time:.4f}s")

Vectorized time (100k rows): 0.0032 seconds

Speed comparison:
Loop:        0.0896s
apply():     0.0991s
Vectorized:  0.0032s


**Example:**

A company wanting to categorize employees as "Senior" or "Junior" based on age could use **apply()** with a custom function, but should prefer vectorized conditions (like **np.where**) when possible for speed.

In [8]:
def categorize(age):
    return "Senior" if age >= 28 else "Junior"
customers["Category"] = customers["Age"].apply(categorize)
customers

,Name,Age,City,Salary,Bonus,Category
0,Hema,25,Chennai,50000,5000.0,Junior
1,Chitra,27,Delhi,55000,5500.0,Junior
2,Koushi,30,Bangalore,61000,6100.0,Senior
3,Subha,28,Mumbai,58000,5800.0,Senior
4,Swathi,24,Chennai,62000,6200.0,Junior


In [9]:
customers["Category_Fast"] = np.where(customers["Age"] >= 28, "Senior", "Junior")
customers

,Name,Age,City,Salary,Bonus,Category,Category_Fast
0,Hema,25,Chennai,50000,5000.0,Junior,Junior
1,Chitra,27,Delhi,55000,5500.0,Junior,Junior
2,Koushi,30,Bangalore,61000,6100.0,Senior,Senior
3,Subha,28,Mumbai,58000,5800.0,Senior,Senior
4,Swathi,24,Chennai,62000,6200.0,Junior,Junior


### AI/ML Usage

During feature engineering, prefer vectorized logic (**np.where**, boolean masks) over **apply()** with custom Python functions whenever possible, especially on large training datasets, since it directly affects preprocessing pipeline speed.

## 3. Memory Optimization

Large DataFrames can consume a lot of memory. Choosing appropriate data types (e.g., smaller integer types) can significantly reduce memory usage.

**Syntax:**

```
DataFrame.info(memory_usage='deep')
DataFrame['col'] = DataFrame['col'].astype('int32')
```

In [10]:
big_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 4 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   Age     1000000 non-null  int32  
 1   Salary  1000000 non-null  int32  
 2   City    1000000 non-null  str    
 3   Bonus   1000000 non-null  float64
dtypes: float64(1), int32(2), str(1)
memory usage: 29.3 MB


In [11]:
big_df_optimized = big_df.copy()
big_df_optimized["Age"] = big_df_optimized["Age"].astype("int8")
big_df_optimized["Salary"] = big_df_optimized["Salary"].astype("int32")
big_df_optimized.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 4 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   Age     1000000 non-null  int8   
 1   Salary  1000000 non-null  int32  
 2   City    1000000 non-null  str    
 3   Bonus   1000000 non-null  float64
dtypes: float64(1), int32(1), int8(1), str(1)
memory usage: 26.5 MB


**Example:**

A company storing employee ages (which never exceed 100) doesn't need a 64-bit integer column — using int8 instead can cut memory usage for that column significantly.

In [12]:
before_mb = big_df.memory_usage(deep=True).sum() / 1024**2
after_mb = big_df_optimized.memory_usage(deep=True).sum() / 1024**2
print(f"Before optimization: {before_mb:.2f} MB")
print(f"After optimization:  {after_mb:.2f} MB")
print(f"Reduction: {(1 - after_mb/before_mb) * 100:.1f}%")

Before optimization: 29.33 MB
After optimization:  26.47 MB
Reduction: 9.8%


### AI/ML Usage

Reducing memory footprint allows larger datasets to fit in RAM, which is critical when training models on large tabular datasets or when working in memory-constrained environments like free-tier cloud notebooks.

## 4. Category Data Type

The **category** dtype is ideal for columns with a limited number of repeating unique values (like City, Gender, or Department). It stores values as integer codes internally instead of repeating full strings, saving memory and speeding up operations like grouping.

**Syntax:**

```
DataFrame['col'] = DataFrame['col'].astype('category')
```

In [13]:
print("Before converting to category:")
print(big_df["City"].memory_usage(deep=True) / 1024**2, "MB")
big_df["City"] = big_df["City"].astype("category")
print("\nAfter converting to category:")
print(big_df["City"].memory_usage(deep=True) / 1024**2, "MB")

Before converting to category:
14.06747817993164 MB

After converting to category:
0.953857421875 MB


**Example:**

A company with a **City** column containing only 4 unique city names repeated a million times can save substantial memory by converting it to a **category** dtype instead of storing the string repeatedly.

In [14]:
customers["City"] = customers["City"].astype("category")
customers.dtypes

Name                  str
Age                 int64
City             category
Salary              int64
Bonus             float64
Category              str
Category_Fast         str
dtype: object

### AI/ML Usage

Categorical dtypes are commonly used before one-hot encoding or label encoding categorical features for machine learning models, and they also speed up **groupby()** operations on large datasets.

## 5. Efficient Pandas Practices

Beyond individual functions, following good habits keeps Pandas code fast and memory-efficient at scale.

| Practice | Why It Helps |
|---|---|
| Prefer vectorized operations over loops/apply() | Avoids slow row-by-row Python execution |
| Use `category` dtype for repeating text values | Reduces memory and speeds up grouping |
| Downcast numeric types (`int64` → `int32`/`int8`) | Reduces memory footprint |
| Read only needed columns with `usecols` in `read_csv()` | Avoids loading unnecessary data |
| Filter/select data early | Reduces data size for subsequent operations |
| Use `.loc`/`.iloc` instead of chained indexing | Avoids `SettingWithCopyWarning` and hidden copies |
| Use `groupby()` instead of manual loops for aggregation | Uses optimized internal C implementations |
| Avoid growing a DataFrame row-by-row in a loop | Rebuilding a DataFrame repeatedly is very slow |

In [14]:
selected_cols = big_df[["Age", "Salary"]]
selected_cols.head()

,Age,Salary
0,56,126467
1,46,49112
2,32,81452
3,25,140582
4,38,97516


**Example:**

A company processing a large employee CSV file with 30 columns, but only needing **Age** and **Salary** for an analysis, should load just those two columns using **usecols** rather than reading the entire file into memory.

In [15]:
avg_salary_by_city = big_df.groupby("City", observed=True)["Salary"].mean()
avg_salary_by_city

City
Bangalore    84932.978025
Chennai      84833.044271
Delhi        85079.852300
Mumbai       85144.791209
Name: Salary, dtype: float64

### AI/ML Usage

Efficient data handling practices directly reduce the time spent in the data preprocessing stage of an ML pipeline, allowing more iterations of model experimentation within the same time and memory budget.